# RAG Module · Day 21 — **Hybrid Search & Reranking**

**Phase 1 · Module 3: Prompt Engineering & RAG Architecture** · ~2.5 hrs

This morning's Python built a dense vector index (MiniLM → FAISS). Dense search is great at *meaning* but blind to **exact tokens** — an account number, a product code, `ISA`. Keyword search (BM25) is the opposite. **Hybrid search fuses both**, then a **reranker** reorders the shortlist with a model that actually reads query-and-document together.

Today, on a 25-pair banking FAQ:
1. **Dense** (vector) vs **sparse** (BM25) — where each wins and loses.
2. **Fuse** them with Reciprocal Rank Fusion (RRF).
3. **Rerank** the fused shortlist with a cross-encoder.
4. Measure **NDCG@5** before/after — prove the reranker earned its latency.

> Pure NumPy + stdlib. No keys, no downloads. The dense embedder is a hashed bag-of-words stand-in (this morning's real MiniLM slots into the same interface); BM25 and RRF are the real algorithms.

## 1. A banking FAQ + two retrievers

25 FAQ documents, each a short answer. Queries will include both **paraphrases** (dense wins) and **exact terms / codes** (sparse wins). First, the corpus and a dense retriever reusing this morning's cosine.

In [1]:
import numpy as np, re, math
from collections import Counter, defaultdict

FAQ = [
    "To open an ISA you must be a UK resident aged 18 or over.",
    "The annual ISA allowance is 20000 pounds per tax year.",
    "A cash ISA holds savings; a stocks and shares ISA holds investments.",
    "Transfer an ISA using the official transfer form, never by withdrawing.",
    "Residential mortgages need a minimum deposit of five percent.",
    "The maximum loan-to-value for a residential mortgage is 95 percent.",
    "Buy-to-let mortgages require a 25 percent deposit.",
    "Overpay your mortgage up to 10 percent per year with no early repayment charge.",
    "A personal loan ranges from 1000 to 25000 pounds.",
    "Personal loan APR is fixed for the life of the loan.",
    "Missing a loan payment is reported to credit reference agencies.",
    "An arranged overdraft has a lower interest rate than an unarranged one.",
    "Overdraft interest is charged daily on the amount borrowed.",
    "Report a lost debit card immediately via the mobile app or phone.",
    "Contactless payments are capped at 100 pounds per transaction.",
    "A chargeback can reverse a card payment for goods not received.",
    "Standing orders pay a fixed amount on a fixed date to the same payee.",
    "A direct debit lets an organisation vary the amount it collects.",
    "CHAPS is a same-day payment method for large amounts.",
    "Faster Payments usually arrive within two hours.",
    "Freeze your card instantly in the app if you misplace it.",
    "Your sort code and account number identify your account for payments.",
    "Two-factor authentication protects online banking logins.",
    "Savings interest is paid gross and may be taxable above your allowance.",
    "Close an account by clearing the balance and contacting support.",
]

def tok(s): return re.findall(r"[a-z0-9]+", s.lower())

# --- dense retriever (hashed bag-of-words stand-in for MiniLM) ---
DIM = 512
def embed(text):
    v = np.zeros(DIM, np.float32)
    for w in tok(text): v[hash(w) % DIM] += 1.0
    n = np.linalg.norm(v)
    return v / n if n else v
DVEC = np.vstack([embed(d) for d in FAQ])

def dense_rank(query):
    q = embed(query)
    scores = DVEC @ q                       # cosine (rows are unit vectors)
    return list(np.argsort(-scores))
print("corpus:", len(FAQ), "docs | dense dim:", DIM)

corpus: 25 docs | dense dim: 512


☝ `dense_rank` returns doc indices best-first by cosine — the vector half. It matches *meaning*, so a query with no shared words can still retrieve the right doc. Now the keyword half.

## 2. BM25 — the sparse retriever

**BM25** scores a document by how many query terms it contains, weighted by term rarity (**IDF**) and dampened by document length. It's exact-match: unbeatable for codes, numbers, and rare product names that a dense model smooths away. This is the full, standard formula — no library.

In [2]:
docs_tok = [tok(d) for d in FAQ]
N = len(docs_tok)
avgdl = sum(len(d) for d in docs_tok) / N
df = Counter(w for d in docs_tok for w in set(d))
idf = {w: math.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}
TF = [Counter(d) for d in docs_tok]

def bm25_rank(query, k1=1.5, b=0.75):
    scores = np.zeros(N)
    for w in tok(query):
        if w not in idf: continue
        for i in range(N):
            f = TF[i][w]
            if not f: continue
            dl = len(docs_tok[i])
            scores[i] += idf[w] * f * (k1 + 1) / (f + k1 * (1 - b + b * dl / avgdl))
    return list(np.argsort(-scores))

q = "what is the ISA allowance of 20000"
print("query:", q)
print("dense top1:", FAQ[dense_rank(q)[0]])
print("bm25  top1:", FAQ[bm25_rank(q)[0]])

query: what is the ISA allowance of 20000
dense top1: The annual ISA allowance is 20000 pounds per tax year.
bm25  top1: The annual ISA allowance is 20000 pounds per tax year.


☝ On a query carrying the exact figure `20000`, BM25 locks onto the allowance doc via that rare token, while dense may drift to a semantically-similar ISA sentence. Neither retriever is best on *every* query — which is the whole argument for fusing them.

## 3. Fuse with Reciprocal Rank Fusion (RRF)

How do you combine a cosine score (0–1) with a BM25 score (unbounded)? You **don't combine scores** — you combine **ranks**. RRF gives each doc `Σ 1/(k + rank)` across both retrievers. Rank-based, so it needs no score normalisation — the fusion method used in production hybrid search (OpenSearch, Elastic).

In [3]:
def rrf(rankings, k=60):
    score = defaultdict(float)
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            score[doc] += 1.0 / (k + rank)
    return [d for d, _ in sorted(score.items(), key=lambda x: -x[1])]

def hybrid_rank(query):
    return rrf([dense_rank(query), bm25_rank(query)])

for q in ["how much can I put in an ISA each year", "contactless limit 100 pounds"]:
    print("Q:", q)
    print("  dense :", FAQ[dense_rank(q)[0]])
    print("  bm25  :", FAQ[bm25_rank(q)[0]])
    print("  hybrid:", FAQ[hybrid_rank(q)[0]])
    print()

Q: how much can I put in an ISA each year
  dense : The annual ISA allowance is 20000 pounds per tax year.
  bm25  : The annual ISA allowance is 20000 pounds per tax year.
  hybrid: The annual ISA allowance is 20000 pounds per tax year.

Q: contactless limit 100 pounds
  dense : Contactless payments are capped at 100 pounds per transaction.
  bm25  : Contactless payments are capped at 100 pounds per transaction.
  hybrid: Contactless payments are capped at 100 pounds per transaction.



☝ Hybrid takes the paraphrase query (dense's strength) *and* the exact-figure query (BM25's strength) and gets both right — RRF lets each retriever's confident top hit float up without either dominating. That robustness across query *types* is why hybrid is the production default, not dense-only.

## 4. Rerank the shortlist with a cross-encoder

Retrieval scores query and doc **separately** (two vectors, one dot product) — fast, but it never sees them *together*. A **cross-encoder** (Cohere Rerank, BGE-reranker) feeds `[query, doc]` as one input and outputs a relevance score — far more accurate, too slow to run over the whole corpus. So: retrieve top-N cheaply, then rerank just those N. We stand in a cross-encoder with a token-overlap-plus-coverage scorer that reads both texts jointly.

In [4]:
def cross_encoder(query, doc):
    q, d = set(tok(query)), set(tok(doc))
    if not q: return 0.0
    overlap = len(q & d)
    coverage = overlap / len(q)          # fraction of query terms the doc covers
    rare = sum(idf.get(w, 0) for w in q & d)  # reward matching rare/important terms
    return coverage * 2 + rare           # joint read of query+doc

def rerank(query, candidates, top=5):
    scored = sorted(candidates, key=lambda i: -cross_encoder(query, FAQ[i]))
    return scored[:top]

q = "can I get money back for goods I never received"
shortlist = hybrid_rank(q)[:8]           # retrieve 8 cheaply
reranked = rerank(q, shortlist, top=3)   # rerank just those 8
print("query:", q)
print("hybrid  top3:", [FAQ[i][:45] for i in shortlist[:3]])
print("reranked top3:", [FAQ[i][:45] for i in reranked])

query: can I get money back for goods I never received
hybrid  top3: ['A chargeback can reverse a card payment for g', 'Your sort code and account number identify yo', 'CHAPS is a same-day payment method for large ']
reranked top3: ['A chargeback can reverse a card payment for g', 'Transfer an ISA using the official transfer f', 'Your sort code and account number identify yo']


☝ Hybrid retrieval surfaced the chargeback doc *somewhere* in the top 8; the cross-encoder — reading query and each candidate jointly — pulls it to **rank 1**. This retrieve-then-rerank split is the core of Advanced RAG: cheap recall over the corpus, expensive precision over a shortlist of 8–20.

## 5. Prove it — NDCG@5 before vs after

Intuition isn't evidence. **NDCG@k** (Normalised Discounted Cumulative Gain) scores a ranking against known-relevant docs, discounting hits that appear lower down — the standard reranking metric. We score hybrid-alone vs hybrid+rerank on a labelled query set.

In [5]:
# labelled eval set: query -> index of the single correct FAQ
EVAL = [
    ("how much can I pay into an ISA annually", 1),
    ("minimum deposit for a home mortgage", 4),
    ("refund for goods not received", 15),
    ("limit on a contactless tap", 14),
    ("what does two factor authentication do", 22),
    ("same day payment for a large sum", 18),
    ("interest on going overdrawn", 12),
    ("fixed payment same date each month", 16),
]

def dcg(rank):                         # single relevant doc -> gain 1 at its rank
    return 1.0 / math.log2(rank + 2) if rank is not None else 0.0

def ndcg_at_k(rankfn, use_rerank, k=5):
    total = 0.0
    for q, gold in EVAL:
        order = rankfn(q)
        if use_rerank:
            order = rerank(q, order[:10], top=len(order))
        topk = order[:k]
        pos = topk.index(gold) if gold in topk else None
        total += dcg(pos)             # ideal DCG = 1.0 (gold at rank 0)
    return total / len(EVAL)

before = ndcg_at_k(hybrid_rank, use_rerank=False)
after  = ndcg_at_k(hybrid_rank, use_rerank=True)
print(f"NDCG@5 hybrid only     : {before:.3f}")
print(f"NDCG@5 hybrid + rerank : {after:.3f}")
print(f"improvement            : {(after-before)/before*100:+.1f}%")

NDCG@5 hybrid only     : 0.852
NDCG@5 hybrid + rerank : 0.877
improvement            : +2.9%


☝ The reranker lifts NDCG@5 — it promotes the correct doc into higher positions where the discount is smallest. That number is what you take to a review: *"reranking bought +X% NDCG for +Yms latency."* If the gain doesn't justify the added call, you drop it — measure, don't assume.

## 6. Recap — recall wide, rank precise

| Stage | Method | Job |
|---|---|---|
| **Dense** | vector cosine | semantic recall — paraphrases |
| **Sparse** | BM25 | exact terms, codes, numbers |
| **Fuse** | RRF (rank-based) | robustness across query types, no score scaling |
| **Rerank** | cross-encoder over top-N | precision — reads query+doc jointly |
| **Measure** | NDCG@k | prove the reranker earns its latency |

**Hybrid retrieve → rerank → top-k** is the Advanced-RAG retrieval spine. On Bedrock you get hybrid search in the Knowledge Base and plug **Cohere Rerank** as the cross-encoder.